# Exploratory Data Analysis: Stack Exchange Data

This notebook explores the structured Stack Exchange data to understand:
- Distribution of questions by tags
- Score distribution
- Text length statistics
- Sample Q&A pairs

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

# Set paths
PROJECT_ROOT = Path.cwd().parent
STRUCTURED_DIR = PROJECT_ROOT / "data" / "01_structured"

print(f"Project root: {PROJECT_ROOT}")

## Load Data

In [ ]:
# Load Stack Exchange data
se_path = STRUCTURED_DIR / "stack_exchange.jsonl"

if se_path.exists():
    records = []
    with open(se_path) as f:
        for line in f:
            records.append(json.loads(line))
    
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records")
    df.head()
else:
    print(f"Data file not found: {se_path}")
    print("Run the structuring script first.")

## Basic Statistics

In [ ]:
if 'df' in dir():
    print("Dataset Shape:", df.shape)
    print("\nColumns:", df.columns.tolist())
    print("\nNull counts:")
    print(df.isnull().sum())

## Tag Distribution

In [ ]:
if 'df' in dir():
    # Flatten tags
    all_tags = []
    for tags in df['tags']:
        if tags:
            all_tags.extend(tags)
    
    tag_counts = Counter(all_tags)
    
    # Plot top 20 tags
    top_tags = tag_counts.most_common(20)
    tags, counts = zip(*top_tags)
    
    plt.figure(figsize=(12, 6))
    plt.barh(range(len(tags)), counts)
    plt.yticks(range(len(tags)), tags)
    plt.xlabel('Count')
    plt.title('Top 20 Tags')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal unique tags: {len(tag_counts)}")

## Text Length Analysis

In [ ]:
if 'df' in dir():
    # Calculate text lengths (word count)
    df['question_words'] = df['question_body'].apply(lambda x: len(x.split()) if x else 0)
    df['answer_words'] = df['answer_body'].apply(lambda x: len(x.split()) if x else 0)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Question length distribution
    axes[0].hist(df['question_words'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Word Count')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Question Length Distribution')
    axes[0].axvline(df['question_words'].median(), color='red', linestyle='--', label=f'Median: {df["question_words"].median():.0f}')
    axes[0].legend()
    
    # Answer length distribution
    axes[1].hist(df['answer_words'], bins=50, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Word Count')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Answer Length Distribution')
    axes[1].axvline(df['answer_words'].median(), color='red', linestyle='--', label=f'Median: {df["answer_words"].median():.0f}')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print("\nText Length Statistics:")
    print(df[['question_words', 'answer_words']].describe())

## Sample Records

In [ ]:
if 'df' in dir():
    # Show a few sample records
    for idx in range(min(3, len(df))):
        record = df.iloc[idx]
        print("=" * 80)
        print(f"Record {idx + 1}")
        print("=" * 80)
        print(f"\nTitle: {record['title']}")
        print(f"Tags: {record['tags']}")
        print(f"\nQuestion ({record['question_words']} words):")
        print(record['question_body'][:500] + "..." if len(record['question_body']) > 500 else record['question_body'])
        print(f"\nAnswer ({record['answer_words']} words):")
        print(record['answer_body'][:500] + "..." if len(record['answer_body']) > 500 else record['answer_body'])
        print()

## Date Distribution (if available)

In [ ]:
if 'df' in dir() and 'creation_date' in df.columns:
    # Parse dates
    df['date'] = pd.to_datetime(df['creation_date'], errors='coerce')
    
    if df['date'].notna().any():
        df['year'] = df['date'].dt.year
        
        plt.figure(figsize=(12, 5))
        df['year'].value_counts().sort_index().plot(kind='bar')
        plt.xlabel('Year')
        plt.ylabel('Number of Questions')
        plt.title('Questions by Year')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## Summary

In [ ]:
if 'df' in dir():
    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Total records: {len(df):,}")
    print(f"Unique tags: {len(tag_counts):,}")
    print(f"Avg question length: {df['question_words'].mean():.1f} words")
    print(f"Avg answer length: {df['answer_words'].mean():.1f} words")
    print(f"\nTop 5 tags: {', '.join([t for t, _ in tag_counts.most_common(5)])}")